<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/Nikhitha/Speech-to-Text%20Pipeline%3A%20Audio%20Conversion%20(16kHz%20Mono)%2C%20VOSK%20%26%20Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q vosk gTTS faster-whisper pydub soundfile sentencepiece
!apt -qq install -y ffmpeg

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.4 MB/s eta 0:00:00
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [2]:
from google.colab import files
from gtts import gTTS
from IPython.display import Audio
import os, wave, json, subprocess
import torch

from vosk import Model, KaldiRecognizer
from faster_whisper import WhisperModel

In [3]:
print("Upload an audio file (mp3 / wav):")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
else:
    tts = gTTS("Hello this is a test audio")
    audio_file = "sample.mp3"
    tts.save(audio_file)

Upload an audio file (mp3 / wav):


Saving sample-3s.mp3 to sample-3s.mp3


In [4]:
wav_file = "audio.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", audio_file, "-ar", "16000", "-ac", "1", wav_file],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Audio converted to WAV")

Audio converted to WAV


In [5]:
if not os.path.exists("vosk-model"):
    !wget -q https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
    !unzip -q vosk-model-small-en-us-0.15.zip
    !mv vosk-model-small-en-us-0.15 vosk-model
    !rm vosk-model-small-en-us-0.15.zip

In [6]:
wf = wave.open(wav_file, "rb")
vosk_model = Model("vosk-model")
rec = KaldiRecognizer(vosk_model, wf.getframerate())

vosk_text = ""

while True:
    data = wf.readframes(4000)
    if not data:
        break
    if rec.AcceptWaveform(data):
        vosk_text += json.loads(rec.Result()).get("text", "") + " "

vosk_text += json.loads(rec.FinalResult()).get("text", "")

print("\n===== VOSK OUTPUT =====")
print(vosk_text.strip())


===== VOSK OUTPUT =====



In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = WhisperModel("small", device=device)

segments, _ = whisper_model.transcribe(wav_file)
whisper_text = " ".join([seg.text for seg in segments])

print("\n===== WHISPER OUTPUT =====")
print(whisper_text.strip())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]


===== WHISPER OUTPUT =====
You
